# The Examiner — BluffBuster Training Notebook

**End-to-end runnable on Google Colab (A100) or HuggingFace Spaces GPU**

Run all cells top-to-bottom on a **clean runtime** with no modifications required.

## Training Architecture
Uses **TRL OpenEnv `environment_factory`** for multi-turn GRPO training:
- Model calls `ask(section_id, question_text)` and `classify(classifications_json)` as function-calling tools
- TRL handles the multi-turn loop, tool-call parsing, and context management automatically
- Pattern: [TRL OpenEnv docs](https://huggingface.co/docs/trl/en/openenv) | [Unsloth 2048 example](https://github.com/meta-pytorch/OpenEnv/tree/main/tutorial/examples)

## Flow
1. Install dependencies (TRL >= 0.9.0 for environment_factory)
2. Set credentials (W&B, HF)
3. Clone repo + import
4. Oracle calibration
5. Baseline evaluation
6. DEBUG smoke test (pipeline health)
7. DEMO training run (TRL OpenEnv)
8. Final evaluation
9. Plot generation
10. Push to HF Hub

In [ ]:
# Cell 1 — Install all dependencies
# TRL >= 0.9.0 required for environment_factory (OpenEnv multi-turn GRPO)
# TRL OpenEnv docs: https://huggingface.co/docs/trl/en/openenv
# Unsloth example: https://github.com/meta-pytorch/OpenEnv/tree/main/tutorial/examples

!pip install -q "unsloth[colab-new]" "trl>=0.9.0" wandb "gradio>=4.0" "pydantic>=2.0" \
    numpy torch transformers accelerate bitsandbytes matplotlib seaborn \
    sentence-transformers datasets huggingface_hub

# Print key versions for reproducibility
import subprocess
for pkg in ["trl", "unsloth", "transformers", "torch"]:
    r = subprocess.run(["pip", "show", pkg], capture_output=True, text=True)
    for line in r.stdout.splitlines():
        if line.startswith("Version"):
            print(f"  {pkg}: {line}")

print("All dependencies installed")

In [ ]:
# Cell 2 — Authentication
# Uses Colab Secrets — do NOT hardcode API keys here.
import os
from google.colab import userdata

os.environ['WANDB_API_KEY'] = userdata.get('WANDB_API_KEY')
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')

import wandb
wandb.login(key=os.environ['WANDB_API_KEY'], relogin=True)

from huggingface_hub import login as hf_login
hf_login(token=os.environ['HF_TOKEN'])

print('✓ W&B and HF authenticated')

In [ ]:
# Cell 3 — Clone repo and set working directory
import subprocess, os

REPO_URL = 'https://github.com/Akshansh0519/BluffBuster_env.git'
REPO_DIR = 'BluffBuster_env'

if not os.path.exists(REPO_DIR):
    subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(['git', '-C', REPO_DIR, 'pull'], check=True)

os.chdir(REPO_DIR)
print(f'✓ Working directory: {os.getcwd()}')

# Ensure outputs dirs exist
os.makedirs('outputs/eval', exist_ok=True)
os.makedirs('outputs/plots', exist_ok=True)
os.makedirs('outputs/transcripts', exist_ok=True)
os.makedirs('outputs/checkpoints', exist_ok=True)
print('✓ Output directories ready')

import sys
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

In [ ]:
# Cell 4 — Oracle Calibration (BLOCKS TRAINING — must pass before proceeding)
# Fits (alpha, beta, gamma) on 200-episode calibration split.
# Target: Brier <= 0.18, terminal accuracy >= 0.75

import json, os

CAL_PATH = 'outputs/eval/oracle_calibration.json'

if os.path.exists(CAL_PATH):
    print('Oracle calibration file already exists — skipping (delete to recalibrate).')
    with open(CAL_PATH) as f:
        cal = json.load(f)
    print(f"  Brier: {cal['calibration_metrics']['mean_brier']:.4f} (target <= 0.18)")
    print(f"  Accuracy: {cal['calibration_metrics']['terminal_accuracy']:.4f} (target >= 0.75)")
    assert cal['calibration_metrics']['mean_brier'] <= 0.18, 'Brier too high — recalibrate!'
    assert cal['calibration_metrics']['terminal_accuracy'] >= 0.75, 'Accuracy too low — recalibrate!'
else:
    from examiner_env.calibration import run_calibration
    from examiner_env.knowledge_base import KB
    result = run_calibration(KB, n_episodes=200, output_path=CAL_PATH)
    print(f"✓ Calibration complete: Brier={result['calibration_metrics']['mean_brier']:.4f}")

print('✓ Oracle calibration gate: PASS')

In [ ]:
# Cell 5 — Baseline Evaluation
# Runs all 3 baselines on the frozen eval suite BEFORE any training.
# Saves outputs/eval/baseline_metrics.json

import json, os
from examiner_env.baselines import RandomExaminer, DefinitionalExaminer, BayesianHeuristicExaminer
from examiner_env.knowledge_base import KB
from training.eval import run_eval

with open('eval_config.json') as f:
    eval_config = json.load(f)

baseline_metrics = {}
examiners = [
    ('RandomExaminer', RandomExaminer()),
    ('DefinitionalExaminer', DefinitionalExaminer()),
    ('BayesianHeuristicExaminer', BayesianHeuristicExaminer(KB)),
]

for name, examiner in examiners:
    print(f'Evaluating {name}...')
    metrics = run_eval(examiner, eval_config, KB)
    baseline_metrics[name] = metrics
    print(f'  accuracy={metrics["classification_accuracy"]:.3f}, '
          f'info_gain={metrics["avg_info_gain_per_turn"]:.4f}, '
          f'ECE={metrics["calibration_ECE"]:.4f}')

with open('outputs/eval/baseline_metrics.json', 'w') as f:
    json.dump(baseline_metrics, f, indent=2)

# Sanity: DefinitionalExaminer should beat RandomExaminer
def_acc = baseline_metrics['DefinitionalExaminer']['classification_accuracy']
rand_acc = baseline_metrics['RandomExaminer']['classification_accuracy']
assert def_acc > rand_acc, f'DefinitionalExaminer ({def_acc:.3f}) should beat Random ({rand_acc:.3f})'

# Sanity: BayesianHeuristic should have higher info gain than Definitional
bay_gain = baseline_metrics['BayesianHeuristicExaminer']['avg_info_gain_per_turn']
def_gain = baseline_metrics['DefinitionalExaminer']['avg_info_gain_per_turn']
print(f'BayesianHeuristic info_gain ({bay_gain:.4f}) > Definitional ({def_gain:.4f}): {bay_gain > def_gain}')

print('✓ Baseline evaluation complete. baseline_metrics.json saved.')

In [ ]:
# Cell 6 — DEBUG Smoke Test
# Verifies pipeline health on DEBUG config (3 sections, 20 episodes, 1.5B model)
# Does NOT assert reward improvement — only pipeline health.

import json, os
from training.config import DEBUG_CONFIG
from training.train_grpo import train

with open('eval_config.json') as f:
    eval_config = json.load(f)

print('Starting DEBUG smoke test...')
print(f'  Model: {DEBUG_CONFIG.model_name}')
print(f'  Sections: {DEBUG_CONFIG.sections}')
print(f'  Episodes: {DEBUG_CONFIG.num_episodes}')

debug_metrics = train(DEBUG_CONFIG, eval_config)

# Smoke test checks
import numpy as np
assert 'reward_mean' in debug_metrics, 'Missing reward_mean in debug metrics'
assert np.isfinite(debug_metrics['reward_mean']), 'reward_mean is not finite!'
assert -2.05 <= debug_metrics['reward_mean'] <= 1.95, 'reward_mean out of bounds!'

print(f'✓ DEBUG smoke test PASSED')
print(f'  reward_mean={debug_metrics["reward_mean"]:.4f}, ECE={debug_metrics["calibration_ECE"]:.4f}')
print('  Note: no improvement expected in 20 episodes — pipeline health confirmed.')

In [ ]:
# Cell 7 — DEMO Training Run (TRL OpenEnv environment_factory pattern)
# Reference: https://huggingface.co/docs/trl/en/openenv
#            https://github.com/meta-pytorch/OpenEnv/tree/main/tutorial/examples
#
# Training uses environment_factory=ExaminerToolEnv (multi-turn tool-calling GRPO).
# The model calls ask() and classify() as tools each episode.
# TRL handles the multi-turn generation + tool-call parsing loop automatically.
#
# Expected runtime: 2-3 hours on Colab A100 / HF Spaces GPU.

import json
from training.config import DEMO_CONFIG
from training.train_grpo import train, ExaminerToolEnv

with open('eval_config.json') as f:
    eval_config = json.load(f)

print('Starting DEMO training run (TRL OpenEnv environment_factory)...')
print(f'  Model:          {DEMO_CONFIG.model_name}')
print(f'  Sections:       {DEMO_CONFIG.sections}')
print(f'  Episodes:       {DEMO_CONFIG.num_episodes}')
print(f'  Training styles:{DEMO_CONFIG.fake_styles_train}')
print(f'  Held-out styles:{DEMO_CONFIG.eval_styles_held_out}')
print(f'  Held-out secs:  {DEMO_CONFIG.held_out_sections}')
print()
print('Environment: ExaminerToolEnv (ask + classify tools)')
print('  ask(section_id, question_text) -> student response')
print('  classify(classifications_json) -> episode result + reward')

demo_metrics = train(DEMO_CONFIG, eval_config)

print()
print('=== DEMO Training Complete ===')
print(f'  Final accuracy (held-out):  {demo_metrics["classification_accuracy"]:.3f}')
print(f'  Final info gain/turn:       {demo_metrics["avg_info_gain_per_turn"]:.4f}')
print(f'  Final ECE:                  {demo_metrics["calibration_ECE"]:.4f}')
print(f'  Final R_total mean:         {demo_metrics["reward_mean"]:.4f}')
print('outputs/eval/final_metrics.json saved')

In [ ]:
# Cell 8 — Final Evaluation Summary
# Loads and prints the comparison table: 4 examiners on held-out eval suite.

import json

with open('outputs/eval/baseline_metrics.json') as f:
    baseline = json.load(f)
with open('outputs/eval/final_metrics.json') as f:
    final = json.load(f)

print('=== Held-Out Eval Suite Comparison Table ===')
print(f'{"Examiner":<30} {"Accuracy":>10} {"InfoGain/T":>12} {"ECE":>8} {"FalseAccus":>12}')
print('-' * 76)

for name in ['RandomExaminer', 'DefinitionalExaminer', 'BayesianHeuristicExaminer']:
    m = baseline.get(name, {})
    print(f'{name:<30} {m.get("classification_accuracy", float("nan")):>10.3f} '
          f'{m.get("avg_info_gain_per_turn", float("nan")):>12.4f} '
          f'{m.get("calibration_ECE", float("nan")):>8.4f} '
          f'{m.get("false_accusation_rate", float("nan")):>12.4f}')

trained = final.get('TrainedExaminer', final)  # final_metrics may be flat
print(f'{"TrainedExaminer (DEMO)":<30} {trained.get("classification_accuracy", float("nan")):>10.3f} '
      f'{trained.get("avg_info_gain_per_turn", float("nan")):>12.4f} '
      f'{trained.get("calibration_ECE", float("nan")):>8.4f} '
      f'{trained.get("false_accusation_rate", float("nan")):>12.4f}')

print('\n(Results on held-out F3 style + S05 section — not seen during training)')

In [ ]:
# Cell 9 — Generate Evidence Plots
# All plots derived from real training data. MSR-3 compliance.

from scripts.select_transcripts import select_transcripts
from scripts.generate_plots import generate_all_plots

# 1. Select behavior-based transcripts
print('Selecting before/after transcripts...')
transcript_result = select_transcripts(
    baseline_path='outputs/eval/baseline_metrics.json',
    final_path='outputs/eval/final_metrics.json',
    eval_config_path='eval_config.json',
    out_dir='outputs/transcripts',
)
print(f'  Selected seed: {transcript_result["episode_seed"]}, R_info_gap: {transcript_result["R_info_gap"]:.4f}')

# 2. Generate all 9 plots
print('\nGenerating evidence plots...')
plot_paths = generate_all_plots(
    checkpoint_metrics_path='outputs/eval/checkpoint_metrics.json',
    baseline_metrics_path='outputs/eval/baseline_metrics.json',
    final_metrics_path='outputs/eval/final_metrics.json',
    after_transcript_path='outputs/transcripts/after_transcript.json',
    config_name='DEMO',
)

# Log W&B run ID to plots README
import wandb
if wandb.run:
    run_id = wandb.run.id
    run_url = wandb.run.get_url()
    with open('outputs/plots/README.md', 'a') as f:
        f.write(f'\n## Training Run Provenance\n')
        f.write(f'- W&B run ID: `{run_id}`\n')
        f.write(f'- W&B run URL: {run_url}\n')
    print(f'✓ W&B run ID logged: {run_id}')

print(f'\n✓ {len(plot_paths)} evidence plots saved to outputs/plots/')

In [ ]:
# Cell 10 — Push artifacts to HuggingFace Hub
# Pushes: eval outputs, plots, transcripts
# Does NOT push video files (MSR-9 compliance)

import os
from huggingface_hub import HfApi

HF_REPO_ID = 'Akshansh0519/BluffBuster_env'  # update if repo name differs
api = HfApi()

def push_folder(local_dir: str, repo_path: str):
    for fname in os.listdir(local_dir):
        fpath = os.path.join(local_dir, fname)
        if os.path.isfile(fpath) and not fname.endswith(('.mp4', '.mov', '.avi', '.mkv')):
            api.upload_file(
                path_or_fileobj=fpath,
                path_in_repo=os.path.join(repo_path, fname),
                repo_id=HF_REPO_ID,
                repo_type='model',
                commit_message=f'Upload {fname}',
            )
            print(f'  ✓ Pushed {repo_path}/{fname}')

print('Pushing eval outputs...')
push_folder('outputs/eval', 'outputs/eval')

print('Pushing plots...')
push_folder('outputs/plots', 'outputs/plots')

print('Pushing transcripts...')
push_folder('outputs/transcripts', 'outputs/transcripts')

print('\n✓ All artifacts pushed to HuggingFace Hub')
print('No video files committed (MSR-9 compliance)')

In [ ]:
# Cell 11 — (Optional) Run on HuggingFace Spaces GPU using HF Credits
#
# If you have HF credits and want to run training on a HF Space GPU instead of
# Colab, use the ZeroGPU or persistent GPU Space approach:
#
# OPTION A: Duplicate this notebook to a HF Space with GPU hardware
#   1. Go to huggingface.co/new-space
#   2. Select SDK: "Jupyter Notebook" (or "Static")
#   3. Set Hardware: A10G or A100 (uses HF credits)
#   4. Upload this notebook, run cells top-to-bottom
#
# OPTION B: Use HF Inference API / AutoTrain (not recommended for custom envs)
#
# OPTION C: ZeroGPU Space — run training via a Gradio app in a Space
#   This requires wrapping the training loop in a Gradio function with
#   @spaces.GPU decorator.
#
# For OPTION C, here is the minimal wrapper pattern:

HF_SPACE_TRAIN_CODE = '''
# hf_space/train_space.py — ZeroGPU training launcher
# Deploy to a HF Space with ZeroGPU hardware, then trigger via the UI.

import gradio as gr
import spaces  # HF ZeroGPU

@spaces.GPU(duration=7200)  # 2 hour GPU allocation
def launch_training(config_name: str, wandb_key: str, hf_token: str) -> str:
    import os, json
    os.environ["WANDB_API_KEY"] = wandb_key
    os.environ["HF_TOKEN"] = hf_token

    from training.config import get_config
    from training.train_grpo import train

    with open("eval_config.json") as f:
        eval_config = json.load(f)

    config = get_config(config_name)
    metrics = train(config, eval_config)
    return f"Training complete. accuracy={metrics['classification_accuracy']:.3f}"

with gr.Blocks() as app:
    gr.Markdown("# BluffBuster Training Launcher")
    config_dd = gr.Dropdown(["DEBUG", "DEMO"], value="DEBUG", label="Config")
    wandb_tb = gr.Textbox(label="W&B API Key", type="password")
    hf_tb = gr.Textbox(label="HF Token", type="password")
    run_btn = gr.Button("Launch Training", variant="primary")
    result = gr.Textbox(label="Result")
    run_btn.click(launch_training, inputs=[config_dd, wandb_tb, hf_tb], outputs=result)

app.launch()
'''

print("HF Space training pattern loaded.")
print("To use HF credits for training:")
print("  1. Create a new HF Space with A10G GPU hardware")
print("  2. Add hf_space/train_space.py to the Space")
print("  3. Set requirements: unsloth[colab-new] trl>=0.9.0 wandb spaces")
print("  4. Open the Space URL, enter your API keys, click Launch Training")
print()
print("Alternatively, use this notebook directly in a HF Space with Jupyter SDK.")
print("HF credits are billed per GPU-hour — check huggingface.co/pricing for rates.")